# **Thông tin nhóm**
- Lớp: ML 23KHDL1
- Nhóm: 6
- Sinh viên:
    - 23127102 - Lê Quang Phúc
    - 23127212 - Nguyễn Quang Đăng Khoa
    - 23127241 - Đoàn Thành Phát
    - 23127332 - Trần Tiến Cường
    - 23127442 - Trầm Hữu Nhân


# **Đánh giá baseline đối với mô hình Tesseract5x**

## Cài đặt thư viện và môi trường 

In [1]:
%%bash
# Cài đặt Tesseract 5.x vào hệ điều hành
add-apt-repository ppa:alex-p/tesseract-ocr5 -y
apt-get update -qq
apt-get install -y -qq tesseract-ocr

# Tải model tiếng Việt
wget https://github.com/tesseract-ocr/tessdata/raw/main/script/Vietnamese.traineddata -O /usr/share/tesseract-ocr/5/tessdata/Vietnamese.traineddata

pip install -q pytesseract Levenshtein pandas tqdm

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,533 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,969 kB]
Get:8 https://ppa.launchpadcontent.net/alex-p/tesseract-ocr5/ubuntu jammy InRelease [18.3 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InR

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2026-04-16 17:51:03--  https://github.com/tesseract-ocr/tessdata/raw/main/script/Vietnamese.traineddata
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/tesseract-ocr/tessdata/main/script/Vietnamese.traineddata [following]
--2026-04-16 17:51:04--  https://raw.githubusercontent.com/tesseract-ocr/tessdata/main/script/Vietnamese.traineddata
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.13

In [ ]:
import os
import json
import unicodedata
import pandas as pd
import Levenshtein
import pytesseract

from pathlib import Path
from tqdm import tqdm
from PIL import Image
from google.colab import drive

# Kết nối Google Drive
drive.mount('/content/drive')

# Đường dẫn
ZIP_PATH = Path('/content/drive/MyDrive/IntroToML - OCR - data/processed_data.zip')
LOCAL_ROOT = Path('/content/local_data')

# Giải nén
if ZIP_PATH.exists():
    if not LOCAL_ROOT.exists():
        !unzip -q "{ZIP_PATH}" -d "{LOCAL_ROOT}"
    else:
        print("Dữ liệu đã có sẵn.")
else:
    print(f"❌ LỖI: Không tìm thấy file {ZIP_PATH}")

TEST_DIR = LOCAL_ROOT / 'test'
print(f"\n Đường dẫn thư mục Test: {TEST_DIR}")

Mounted at /content/drive

 Đường dẫn thư mục Test: /content/local_data/test


## Các hàm chức năng

In [3]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    return text.strip().lower()

def calculate_cer(pred: str, gt: str) -> float:
    pred = normalize_text(pred)
    gt = normalize_text(gt)

    if len(gt) == 0:
        return 1.0 if len(pred) > 0 else 0.0

    edit_dist = Levenshtein.distance(pred, gt)
    cer = edit_dist / len(gt)
    return cer

## Khởi tạo mô hình

In [5]:
# Cấu hình Tesseract
print("Đang nạp mô hình Teseract...")
TESSERACT_LANG = "Vietnamese"

# Quét qua tất cả thư mục con
all_test_samples = []
subfolders = [f for f in TEST_DIR.iterdir() if f.is_dir()]

for subfolder in subfolders:
    label_file = subfolder / 'label.json'
    if not label_file.exists():
        continue

    with open(label_file, 'r', encoding='utf-8') as f:
        ground_truths = json.load(f)

    for img_name, gt_text in ground_truths.items():
        img_path = subfolder / img_name
        if img_path.exists():
            all_test_samples.append((img_path, gt_text))

print(f"Thực hiện đánh giá trên {len(all_test_samples)} ảnh \n")

# Vòng lặp đánh giá
total_cer = 0.0
error_logs = []

for img_path, gt_text in tqdm(all_test_samples, desc="Đang đánh giá"):
    try:
        img = Image.open(str(img_path))

        width, height = img.size
        aspect_ratio = width / height

        # Dòng
        if aspect_ratio > 2.5:
            config_mode = r'--psm 7'
        else:
            # Đoạn
            config_mode = r'--psm 6'

        # Suy luận bằng Tesseract
        pred_text = pytesseract.image_to_string(img, lang=TESSERACT_LANG, config=config_mode)
    except Exception as e:
        pred_text = ""

    # Tính điểm
    cer_score = calculate_cer(pred_text, gt_text)
    total_cer += cer_score

    # Ghi log ảnh lỗi
    if cer_score > 0:
        error_logs.append({
            "folder": img_path.parent.name,
            "image": img_path.name,
            "ground_truth": normalize_text(gt_text),
            "prediction": normalize_text(pred_text),
            "cer_score": round(cer_score, 4)
        })

# Tính toán CER trung bình
test_samples = len(all_test_samples)
average_cer = total_cer / test_samples if test_samples > 0 else 0

print(f"\nCER trung bình: {average_cer * 100:.2f} %")
print(f"Số lượng ảnh dự đoán sai: {len(error_logs)}")

Đang nạp mô hình Teseract...
Thực hiện đánh giá trên 15000 ảnh 



Đang đánh giá: 100%|██████████| 15000/15000 [40:23<00:00,  6.19it/s]


CER trung bình: 98.59 %
Số lượng ảnh dự đoán sai: 14741


## Lữu trữ

In [6]:
# Chuyển đổi danh sách lỗi thành Pandas DataFrame
df_errors = pd.DataFrame(error_logs)

# Sắp xếp từ lỗi nặng nhất xuống nhẹ nhất
df_errors = df_errors.sort_values(by="cer_score", ascending=False)

# Lưu file nội bộ
report_path = LOCAL_ROOT / 'baseline_Tesseract_report.csv'
df_errors.to_csv(report_path, index=False, encoding='utf-8-sig')

# Copy sang Google Drive
drive_report_path = Path('/content/drive/MyDrive/IntroToML - OCR - data/baseline_Tesseract_report.csv')
!cp "{report_path}" "{drive_report_path}"

print(f"Đã lưu lại report của baseline_Tesseract")


Đã lưu lại report của baseline_Tesseract
